In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
import pandas as pd
import numpy as np

print("Libraries imported.")

Libraries imported.


In [3]:
data_path = (
    "/content/drive/MyDrive/"
    "AutoSenseAI/data/duration/"
    "service_duration_benchmark.csv"
)

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (314, 11)


,fix_id,code_id,fix_title,probability_rank,difficulty_level,est_parts_cost_min_usd,est_parts_cost_max_usd,parts_cost_mid_usd,parts_cost_range_usd,est_labor_hours,labor_minutes
0,849,213,Inspect wheel speed sensor connector and harne...,1,Easy DIY,5.0,25.0,15.0,20.0,0.4,24.0
1,850,213,Test wheel speed sensor resistance and signal,2,Moderate DIY,10.0,40.0,25.0,30.0,0.6,36.0
2,851,213,Replace wheel speed sensor,3,Moderate DIY,30.0,120.0,75.0,90.0,0.6,36.0
3,852,213,Repair chafed wheel speed sensor harness wiring,4,Professional Required,15.0,65.0,40.0,50.0,1.2,72.0
4,865,217,Inspect wheel speed sensor connector and harne...,1,Easy DIY,5.0,25.0,15.0,20.0,0.4,24.0


In [4]:
task_catalog = (
    df
    .groupby("fix_title", as_index=False)
    .agg({
        "difficulty_level": "first",
        "probability_rank": "median",
        "est_parts_cost_min_usd": "median",
        "est_parts_cost_max_usd": "median",
        "parts_cost_mid_usd": "median",
        "parts_cost_range_usd": "median",
        "est_labor_hours": "median",
        "labor_minutes": "median"
    })
)

print("Unique task catalogue size:", len(task_catalog))

task_catalog.head()

Unique task catalogue size: 174


,fix_title,difficulty_level,probability_rank,est_parts_cost_min_usd,est_parts_cost_max_usd,parts_cost_mid_usd,parts_cost_range_usd,est_labor_hours,labor_minutes
0,Address oil or coolant consumption poisoning t...,Professional Required,4.0,100.0,900.0,500.0,800.0,3.0,180.0
1,Adjust brake pedal switch plunger clearance,Easy DIY,1.0,3.0,15.0,9.0,12.0,0.3,18.0
2,Adjust transmission range sensor,Moderate DIY,1.0,15.0,40.0,27.5,25.0,0.7,42.0
3,Change engine oil and filter to spec viscosity,Easy DIY,1.0,25.0,60.0,42.5,35.0,0.5,30.0
4,Check HV battery state of charge,Professional Required,1.0,5.0,40.0,22.5,35.0,0.5,30.0


In [5]:
CATEGORY_RULES = {

    "Brakes": [
        r"\bbrake\b",
        r"\bwheel speed sensor\b",
        r"\babs\b",
        r"\bstop lamp\b"
    ],

    "Battery": [
        r"\bbattery\b",
        r"\bcharging\b",
        r"\balternator\b",
        r"\bparasitic draw\b"
    ],

    "Transmission": [
        r"\btransmission\b",
        r"\bgear\b",
        r"\bshift\b",
        r"\btorque converter\b",
        r"\bvehicle speed sensor\b",
        r"\bvss\b"
    ],

    "Engine": [
        r"\bengine\b",
        r"\boil\b",
        r"\bcamshaft\b",
        r"\bcrankshaft\b",
        r"\btiming\b",
        r"\bspark plug\b",
        r"\bignition coil\b",
        r"\bfuel\b",
        r"\binjector\b",
        r"\bmaf\b",
        r"\bmap sensor\b",
        r"\boxygen sensor\b",
        r"\bo2 sensor\b",
        r"\bcoolant\b",
        r"\bthermostat\b",
        r"\bwater pump\b",
        r"\bintake\b",
        r"\bthrottle\b"
    ],

    "Electrical": [
        r"\bwiring\b",
        r"\bcircuit\b",
        r"\bconnector\b",
        r"\bfuse\b",
        r"\brelay\b",
        r"\bsensor\b",
        r"\bmodule\b",
        r"\becm\b",
        r"\becu\b"
    ]
}

In [6]:
def assign_category(title):

    title_lower = str(title).lower()

    for category, patterns in CATEGORY_RULES.items():

        for pattern in patterns:

            if re.search(
                pattern,
                title_lower,
                flags=re.IGNORECASE
            ):
                return category

    return "Unmapped"

In [7]:
task_catalog["bridge_category"] = (
    task_catalog["fix_title"]
    .apply(assign_category)
)

task_catalog[
    [
        "fix_title",
        "bridge_category"
    ]
].head(20)

,fix_title,bridge_category
0,Address oil or coolant consumption poisoning t...,Engine
1,Adjust brake pedal switch plunger clearance,Brakes
2,Adjust transmission range sensor,Transmission
3,Change engine oil and filter to spec viscosity,Engine
4,Check HV battery state of charge,Battery
5,Check coolant level and inspect for external l...,Engine
6,Check engine oil level and condition,Engine
7,"Check for a missing, damaged, or off-track fue...",Engine
8,Check the module's dedicated fuse and battery/...,Battery
9,Check transmission fluid level and condition,Transmission


In [8]:
category_counts = (
    task_catalog[
        "bridge_category"
    ]
    .value_counts()
)

category_counts

,count
bridge_category,
Engine,65
Electrical,45
Unmapped,37
Transmission,12
Brakes,8
Battery,7


In [9]:
mapped_count = (
    task_catalog["bridge_category"]
    != "Unmapped"
).sum()

coverage = (
    mapped_count
    / len(task_catalog)
    * 100
)

print(
    "Mapped tasks:",
    mapped_count
)

print(
    "Total tasks:",
    len(task_catalog)
)

print(
    "Mapping coverage:",
    round(coverage, 2),
    "%"
)

Mapped tasks: 137
Total tasks: 174
Mapping coverage: 78.74 %


In [10]:
for category in [
    "Battery",
    "Brakes",
    "Electrical",
    "Engine",
    "Transmission",
    "Unmapped"
]:

    print("\n")
    print("=" * 70)
    print(category)
    print("=" * 70)

    display(
        task_catalog.loc[
            task_catalog[
                "bridge_category"
            ] == category,
            [
                "fix_title",
                "difficulty_level",
                "labor_minutes"
            ]
        ].head(30)
    )



Battery


,fix_title,difficulty_level,labor_minutes
4,Check HV battery state of charge,Professional Required,30.0
8,Check the module's dedicated fuse and battery/...,Easy DIY,24.0
33,Diagnose parasitic draw or replace alternator,Professional Required,90.0
161,Test battery condition and terminal connections,Easy DIY,24.0
162,Test charging system output under load,Easy DIY,30.0
169,Verify battery and charging health before the ...,Easy DIY,18.0
170,Verify battery/charging voltage before further...,Easy DIY,18.0




Brakes


,fix_title,difficulty_level,labor_minutes
1,Adjust brake pedal switch plunger clearance,Easy DIY,18.0
55,Inspect brake pedal switch connector,Easy DIY,18.0
68,Inspect wheel speed sensor connector and harne...,Easy DIY,24.0
84,Repair chafed wheel speed sensor harness wiring,Professional Required,72.0
97,Repair stop lamp/brake switch circuit wiring,Professional Required,60.0
112,Replace brake pedal switch,Moderate DIY,30.0
145,Replace wheel speed sensor,Moderate DIY,36.0
167,Test wheel speed sensor resistance and signal,Moderate DIY,36.0




Electrical


...

,fix_title,difficulty_level,labor_minutes
0,Address oil or coolant consumption poisoning t...,Professional Required,180.0
3,Change engine oil and filter to spec viscosity,Easy DIY,30.0
5,Check coolant level and inspect for external l...,Easy DIY,30.0
6,Check engine oil level and condition,Easy DIY,18.0
7,"Check for a missing, damaged, or off-track fue...",Easy DIY,12.0
11,Clean MAF sensor and inspect for false-high ai...,Easy DIY,24.0
12,Clean and reseat corroded throttle body connector,Easy DIY,18.0
16,Clean the VTEC oil strainer and test the spool...,Moderate DIY,60.0
17,Clean throttle body and IAC passage,Easy DIY,30.0
18,Clean throttle body and check MAF/MAP sensor r...,Easy DIY,42.0




Transmission


,fix_title,difficulty_level,labor_minutes
2,Adjust transmission range sensor,Moderate DIY,42.0
9,Check transmission fluid level and condition,Easy DIY,30.0
10,Check transmission fluid level and filter cond...,Easy DIY,42.0
24,Command the named shift solenoid with a scan-t...,Moderate DIY,30.0
30,Diagnose and repair VSS harness wiring,Professional Required,72.0
90,Repair internal/external transmission wiring h...,Professional Required,120.0
93,Repair or replace VSS connector and pigtail,Easy DIY,30.0
96,Repair shift solenoid wiring at the valve body...,Professional Required,90.0
143,Replace transmission range sensor,Moderate DIY,60.0
144,Replace vehicle speed sensor,Moderate DIY,36.0




Unmapped


,fix_title,difficulty_level,labor_minutes
13,Clean or replace a damaged reluctor wheel/tone...,Professional Required,150.0
14,Clean or replace a restricted charcoal canister,Moderate DIY,72.0
15,Clean the EGR valve and passages of carbon bui...,Moderate DIY,90.0
22,Clear debris or ice blocking the vent valve,Easy DIY,30.0
23,Command the TCC solenoid with a scan-tool actu...,Moderate DIY,30.0
25,Complete the manufacturer drive cycle,Easy DIY,90.0
29,Diagnose a monitor that never completes,Professional Required,60.0
36,Identify the theft-deterrent generation and ru...,Easy DIY,30.0
37,Inspect AIR delivery hoses and check valves,Easy DIY,42.0
41,Inspect and clean the EGR cooler for restriction,Moderate DIY,120.0


In [11]:
MANUAL_OVERRIDES = {
    # Example:
    # "Replace brake pedal switch": "Brakes",
}

In [12]:
def apply_override(row):

    title = row["fix_title"]

    if title in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[title]

    return row["bridge_category"]

In [13]:
task_catalog["final_category"] = (
    task_catalog.apply(
        apply_override,
        axis=1
    )
)

In [14]:
def get_tasks_for_category(
    predicted_category,
    max_tasks=10
):

    matches = task_catalog[
        task_catalog[
            "final_category"
        ] == predicted_category
    ].copy()

    matches = matches.sort_values(
        [
            "probability_rank",
            "est_labor_hours"
        ],
        ascending=[
            True,
            True
        ]
    )

    return matches.head(
        max_tasks
    )

In [15]:
brake_tasks = get_tasks_for_category(
    "Brakes",
    max_tasks=10
)

display(
    brake_tasks[
        [
            "fix_title",
            "difficulty_level",
            "probability_rank",
            "est_parts_cost_min_usd",
            "est_parts_cost_max_usd",
            "labor_minutes"
        ]
    ]
)

,fix_title,difficulty_level,probability_rank,est_parts_cost_min_usd,est_parts_cost_max_usd,labor_minutes
1,Adjust brake pedal switch plunger clearance,Easy DIY,1.0,3.0,15.0,18.0
68,Inspect wheel speed sensor connector and harne...,Easy DIY,1.0,5.0,25.0,24.0
55,Inspect brake pedal switch connector,Easy DIY,2.0,5.0,20.0,18.0
167,Test wheel speed sensor resistance and signal,Moderate DIY,2.0,10.0,40.0,36.0
112,Replace brake pedal switch,Moderate DIY,3.0,15.0,60.0,30.0
145,Replace wheel speed sensor,Moderate DIY,3.0,30.0,120.0,36.0
97,Repair stop lamp/brake switch circuit wiring,Professional Required,4.0,15.0,60.0,60.0
84,Repair chafed wheel speed sensor harness wiring,Professional Required,4.0,15.0,65.0,72.0


In [16]:
battery_tasks = get_tasks_for_category(
    "Battery",
    max_tasks=10
)

display(
    battery_tasks[
        [
            "fix_title",
            "difficulty_level",
            "probability_rank",
            "est_parts_cost_min_usd",
            "est_parts_cost_max_usd",
            "labor_minutes"
        ]
    ]
)

,fix_title,difficulty_level,probability_rank,est_parts_cost_min_usd,est_parts_cost_max_usd,labor_minutes
170,Verify battery/charging voltage before further...,Easy DIY,1.0,5.0,25.0,18.0
8,Check the module's dedicated fuse and battery/...,Easy DIY,1.0,8.0,25.0,24.0
161,Test battery condition and terminal connections,Easy DIY,1.0,5.0,25.0,24.0
4,Check HV battery state of charge,Professional Required,1.0,5.0,40.0,30.0
169,Verify battery and charging health before the ...,Easy DIY,2.0,5.0,25.0,18.0
162,Test charging system output under load,Easy DIY,2.0,5.0,30.0,30.0
33,Diagnose parasitic draw or replace alternator,Professional Required,4.0,40.0,350.0,90.0


In [17]:
predicted_category = "Brakes"
category_model_score = 0.91

In [18]:
candidate_tasks = get_tasks_for_category(
    predicted_category,
    max_tasks=8
)

print(
    "Predicted category:",
    predicted_category
)

print(
    "Category model score:",
    category_model_score
)

display(
    candidate_tasks[
        [
            "fix_title",
            "difficulty_level",
            "est_parts_cost_min_usd",
            "est_parts_cost_max_usd"
        ]
    ]
)

Predicted category: Brakes
Category model score: 0.91


,fix_title,difficulty_level,est_parts_cost_min_usd,est_parts_cost_max_usd
1,Adjust brake pedal switch plunger clearance,Easy DIY,3.0,15.0
68,Inspect wheel speed sensor connector and harne...,Easy DIY,5.0,25.0
55,Inspect brake pedal switch connector,Easy DIY,5.0,20.0
167,Test wheel speed sensor resistance and signal,Moderate DIY,10.0,40.0
112,Replace brake pedal switch,Moderate DIY,15.0,60.0
145,Replace wheel speed sensor,Moderate DIY,30.0,120.0
97,Repair stop lamp/brake switch circuit wiring,Professional Required,15.0,60.0
84,Repair chafed wheel speed sensor harness wiring,Professional Required,15.0,65.0


In [19]:
selected_task_title = (
    candidate_tasks.iloc[0][
        "fix_title"
    ]
)

print(
    "Technician selected:",
    selected_task_title
)

Technician selected: Adjust brake pedal switch plunger clearance


In [20]:
selected_task = task_catalog[
    task_catalog["fix_title"]
    == selected_task_title
].iloc[0]

selected_task

,1
fix_title,Adjust brake pedal switch plunger clearance
difficulty_level,Easy DIY
probability_rank,1.0
est_parts_cost_min_usd,3.0
est_parts_cost_max_usd,15.0
parts_cost_mid_usd,9.0
parts_cost_range_usd,12.0
est_labor_hours,0.3
labor_minutes,18.0
bridge_category,Brakes


In [21]:
duration_input = pd.DataFrame([
    {
        "fix_title":
            selected_task[
                "fix_title"
            ],

        "difficulty_level":
            selected_task[
                "difficulty_level"
            ],

        "probability_rank":
            selected_task[
                "probability_rank"
            ],

        "parts_cost_mid_usd":
            selected_task[
                "parts_cost_mid_usd"
            ],

        "parts_cost_range_usd":
            selected_task[
                "parts_cost_range_usd"
            ]
    }
])

duration_input

,fix_title,difficulty_level,probability_rank,parts_cost_mid_usd,parts_cost_range_usd
0,Adjust brake pedal switch plunger clearance,Easy DIY,1.0,9.0,12.0


In [22]:
output_dir = (
    "/content/drive/MyDrive/"
    "AutoSenseAI/data/duration"
)

os.makedirs(
    output_dir,
    exist_ok=True
)

catalog_path = os.path.join(
    output_dir,
    "maintenance_task_catalog.csv"
)

task_catalog.to_csv(
    catalog_path,
    index=False
)

print(
    "Saved:",
    catalog_path
)

Saved: /content/drive/MyDrive/AutoSenseAI/data/duration/maintenance_task_catalog.csv


In [23]:
app_catalog = task_catalog[
    [
        "fix_title",
        "final_category",
        "difficulty_level",
        "probability_rank",
        "est_parts_cost_min_usd",
        "est_parts_cost_max_usd",
        "parts_cost_mid_usd",
        "parts_cost_range_usd"
    ]
].copy()

In [24]:
app_catalog_path = os.path.join(
    output_dir,
    "maintenance_task_catalog_app.csv"
)

app_catalog.to_csv(
    app_catalog_path,
    index=False
)

print(
    "Saved:",
    app_catalog_path
)

Saved: /content/drive/MyDrive/AutoSenseAI/data/duration/maintenance_task_catalog_app.csv


In [25]:
print(
    "Duplicate task titles:",
    app_catalog[
        "fix_title"
    ].duplicated().sum()
)

print(
    "Missing category:",
    app_catalog[
        "final_category"
    ].isna().sum()
)

print(
    "Unmapped tasks:",
    (
        app_catalog[
            "final_category"
        ] == "Unmapped"
    ).sum()
)

Duplicate task titles: 0
Missing category: 0
Unmapped tasks: 37


In [26]:
def create_task_bridge_response(
    predicted_category,
    category_model_score,
    max_tasks=5
):

    candidates = get_tasks_for_category(
        predicted_category,
        max_tasks=max_tasks
    )

    return {
        "predictedCategory":
            predicted_category,

        "categoryModelScore":
            round(
                float(
                    category_model_score
                ),
                4
            ),

        "taskSelectionRequired":
            True,

        "candidateTasks":
            candidates[
                [
                    "fix_title",
                    "difficulty_level",
                    "est_parts_cost_min_usd",
                    "est_parts_cost_max_usd"
                ]
            ].to_dict(
                orient="records"
            )
    }

In [27]:
demo_response = (
    create_task_bridge_response(
        predicted_category="Brakes",
        category_model_score=0.91,
        max_tasks=5
    )
)

demo_response

{'predictedCategory': 'Brakes',
 'categoryModelScore': 0.91,
 'taskSelectionRequired': True,
 'candidateTasks': [{'fix_title': 'Adjust brake pedal switch plunger clearance',
   'difficulty_level': 'Easy DIY',
   'est_parts_cost_min_usd': 3.0,
   'est_parts_cost_max_usd': 15.0},
  {'fix_title': 'Inspect wheel speed sensor connector and harness routing',
   'difficulty_level': 'Easy DIY',
   'est_parts_cost_min_usd': 5.0,
   'est_parts_cost_max_usd': 25.0},
  {'fix_title': 'Inspect brake pedal switch connector',
   'difficulty_level': 'Easy DIY',
   'est_parts_cost_min_usd': 5.0,
   'est_parts_cost_max_usd': 20.0},
  {'fix_title': 'Test wheel speed sensor resistance and signal',
   'difficulty_level': 'Moderate DIY',
   'est_parts_cost_min_usd': 10.0,
   'est_parts_cost_max_usd': 40.0},
  {'fix_title': 'Replace brake pedal switch',
   'difficulty_level': 'Moderate DIY',
   'est_parts_cost_min_usd': 15.0,
   'est_parts_cost_max_usd': 60.0}]}

In [28]:
test_unknown = (
    create_task_bridge_response(
        predicted_category="Unknown",
        category_model_score=0.4,
        max_tasks=5
    )
)

test_unknown

{'predictedCategory': 'Unknown',
 'categoryModelScore': 0.4,
 'taskSelectionRequired': True,
 'candidateTasks': []}

## Specific Maintenance Task Bridge

Purpose:
Connect the existing maintenance-category classifier with the
service-duration regression model.

Existing classifier output:
- Battery
- Brakes
- Electrical
- Engine
- Transmission

Duration-model requirement:
The duration regressor requires a specific repair task rather than
only a broad maintenance category.

Bridge Design:
A transparent rule-assisted task catalogue is used to filter repair
tasks by the predicted maintenance category.

The technician then confirms the specific repair task before the
duration model is executed.

Important:
The duration dataset does not provide ground-truth labels matching the
five maintenance categories used by the category classifier.

Therefore, category-to-task mappings are treated as routing heuristics,
not as learned or validated classification labels.

Workflow:

Vehicle information
→ Maintenance category classifier
→ Predicted broad category
→ Candidate repair-task catalogue
→ Technician confirms specific task
→ Random Forest duration regressor
→ Estimated labor duration

This preserves technician oversight and avoids generating unsupported
specific repair decisions automatically.